In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Data Quality Validation — Step 01: Verify Silver Source Tables
#
# Purpose:
# Confirm that all 21 validated Silver Delta tables are available
# before executing the formal data-quality control framework.
#
# This notebook reads from Silver only.
# No Silver source data is modified.

silver_catalog = "workspace"
silver_schema = "silver_sop"

expected_silver_tables = 21
expected_silver_records = 4559

print("DATA QUALITY VALIDATION — SILVER SOURCE VERIFICATION")
print("=" * 100)

# Discover persisted Silver tables
silver_tables = (
    spark.sql(
        f"SHOW TABLES IN {silver_catalog}.{silver_schema}"
    )
    .select("tableName")
    .collect()
)

silver_table_names = sorted(
    row["tableName"]
    for row in silver_tables
)

print(f"Source schema: {silver_catalog}.{silver_schema}")
print(f"Silver tables found: {len(silver_table_names)}")
print()

for i, table_name in enumerate(
    silver_table_names,
    start=1
):
    print(f"{i:>2}. {table_name}")


# ---------------------------------------------------------------
# Validate table count
# ---------------------------------------------------------------

table_count_status = (
    "PASS"
    if len(silver_table_names) == expected_silver_tables
    else "FAIL"
)

print()
print("-" * 100)
print(
    f"Expected Silver tables: {expected_silver_tables}"
)
print(
    f"Actual Silver tables:   {len(silver_table_names)}"
)
print(
    f"Table-count control:    {table_count_status}"
)


# ---------------------------------------------------------------
# Validate total persisted record count
# ---------------------------------------------------------------

total_silver_records = sum(
    spark.table(
        f"{silver_catalog}.{silver_schema}.{table_name}"
    ).count()
    for table_name in silver_table_names
)

record_count_status = (
    "PASS"
    if total_silver_records == expected_silver_records
    else "FAIL"
)

print()
print(
    f"Expected Silver records: {expected_silver_records}"
)
print(
    f"Actual Silver records:   {total_silver_records}"
)
print(
    f"Record-count control:     {record_count_status}"
)


# ---------------------------------------------------------------
# Final source verification
# ---------------------------------------------------------------

print("=" * 100)

if (
    table_count_status == "PASS"
    and record_count_status == "PASS"
):
    print(
        "SOURCE VERIFICATION PASSED: "
        "21 persisted Silver Delta tables and "
        "4,559 records are available for formal "
        "data-quality validation."
    )
else:
    raise ValueError(
        "SOURCE VERIFICATION FAILED. "
        "Do not continue data-quality validation "
        "until the Silver source is reconciled."
    )

DATA QUALITY VALIDATION — SILVER SOURCE VERIFICATION
Source schema: workspace.silver_sop
Silver tables found: 21

 1. alternative_machine
 2. attainment_parameter
 3. customer_order
 4. demand_plan
 5. finished_product_economics
 6. finished_product_type
 7. machine
 8. machine_capacity
 9. packing_capacity
10. packing_machine
11. planning_parameters
12. planning_periods
13. product_inventory
14. product_lot_size
15. product_packing_machine
16. product_production_alternative
17. product_raw_material_requirement
18. product_successor
19. raw_material
20. semi_finished_acquisition
21. semi_finished_product_cost

----------------------------------------------------------------------------------------------------
Expected Silver tables: 21
Actual Silver tables:   21
Table-count control:    PASS

Expected Silver records: 4559
Actual Silver records:   4559
Record-count control:     PASS
SOURCE VERIFICATION PASSED: 21 persisted Silver Delta tables and 4,559 records are available for formal da

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Data Quality Validation — Step 02: Define Data Quality Rule Registry
#
# Purpose:
# Establish a centralized registry of formal data-quality controls
# that will be executed against the persisted Silver Delta tables.
#
# Rule design is based only on structures and relationships validated
# during the Silver transformation process.
#
# Severity:
# CRITICAL = failure can compromise relational or analytical integrity
# WARNING  = failure requires investigation but does not automatically
#            invalidate the entire dataset

print("DATA QUALITY VALIDATION — RULE REGISTRY")
print("=" * 120)


# ---------------------------------------------------------------
# 1. Define formal data-quality rules
# ---------------------------------------------------------------

dq_rules = [

    # -----------------------------------------------------------
    # COMPLETENESS
    # Required analytical identifiers must not be NULL.
    # -----------------------------------------------------------

    {
        "rule_id": "DQ-C001",
        "dimension": "Completeness",
        "table_name": "customer_order",
        "column_name": "order_id",
        "rule_type": "NOT_NULL",
        "severity": "CRITICAL",
        "description": "Customer order identifier must not be NULL."
    },

    {
        "rule_id": "DQ-C002",
        "dimension": "Completeness",
        "table_name": "customer_order",
        "column_name": "finished_product",
        "rule_type": "NOT_NULL",
        "severity": "CRITICAL",
        "description": "Customer order finished-product identifier must not be NULL."
    },

    {
        "rule_id": "DQ-C003",
        "dimension": "Completeness",
        "table_name": "demand_plan",
        "column_name": "finished_product",
        "rule_type": "NOT_NULL",
        "severity": "CRITICAL",
        "description": "Demand-plan finished-product identifier must not be NULL."
    },

    {
        "rule_id": "DQ-C004",
        "dimension": "Completeness",
        "table_name": "demand_plan",
        "column_name": "period",
        "rule_type": "NOT_NULL",
        "severity": "CRITICAL",
        "description": "Demand-plan period must not be NULL."
    },


    # -----------------------------------------------------------
    # UNIQUENESS
    # Business keys validated during Silver must remain unique.
    # -----------------------------------------------------------

    {
        "rule_id": "DQ-U001",
        "dimension": "Uniqueness",
        "table_name": "customer_order",
        "column_name": "order_id",
        "rule_type": "UNIQUE",
        "severity": "CRITICAL",
        "description": "Customer order identifier must remain unique."
    },

    {
        "rule_id": "DQ-U002",
        "dimension": "Uniqueness",
        "table_name": "finished_product_type",
        "column_name": "finished_product",
        "rule_type": "UNIQUE",
        "severity": "CRITICAL",
        "description": "Finished-product identifier must remain unique."
    },

    {
        "rule_id": "DQ-U003",
        "dimension": "Uniqueness",
        "table_name": "machine",
        "column_name": "machine",
        "rule_type": "UNIQUE",
        "severity": "CRITICAL",
        "description": "Machine identifier must remain unique."
    },

    {
        "rule_id": "DQ-U004",
        "dimension": "Uniqueness",
        "table_name": "packing_machine",
        "column_name": "packing_machine",
        "rule_type": "UNIQUE",
        "severity": "CRITICAL",
        "description": "Packing-machine identifier must remain unique."
    },

    {
        "rule_id": "DQ-U005",
        "dimension": "Uniqueness",
        "table_name": "planning_periods",
        "column_name": "period",
        "rule_type": "UNIQUE",
        "severity": "CRITICAL",
        "description": "Planning-period identifier must remain unique."
    },


    # -----------------------------------------------------------
    # REFERENTIAL INTEGRITY
    # Foreign-key-like relationships validated in Silver.
    # -----------------------------------------------------------

    {
        "rule_id": "DQ-R001",
        "dimension": "Referential Integrity",
        "table_name": "customer_order",
        "column_name": "finished_product",
        "rule_type": "FOREIGN_KEY",
        "severity": "CRITICAL",
        "description":
            "Customer-order products must exist in finished_product_type."
    },

    {
        "rule_id": "DQ-R002",
        "dimension": "Referential Integrity",
        "table_name": "demand_plan",
        "column_name": "finished_product",
        "rule_type": "FOREIGN_KEY",
        "severity": "CRITICAL",
        "description":
            "Demand-plan products must exist in finished_product_type."
    },

    {
        "rule_id": "DQ-R003",
        "dimension": "Referential Integrity",
        "table_name": "machine_capacity",
        "column_name": "machine",
        "rule_type": "FOREIGN_KEY",
        "severity": "CRITICAL",
        "description":
            "Machine-capacity machine identifiers must exist in machine."
    },

    {
        "rule_id": "DQ-R004",
        "dimension": "Referential Integrity",
        "table_name": "packing_capacity",
        "column_name": "packing_machine",
        "rule_type": "FOREIGN_KEY",
        "severity": "CRITICAL",
        "description":
            "Packing-capacity machine identifiers must exist in packing_machine."
    },

    {
        "rule_id": "DQ-R005",
        "dimension": "Referential Integrity",
        "table_name": "demand_plan",
        "column_name": "period",
        "rule_type": "FOREIGN_KEY",
        "severity": "CRITICAL",
        "description":
            "Demand-plan periods must exist in planning_periods."
    },


    # -----------------------------------------------------------
    # VALIDITY
    # Domains and structural conditions established in Silver.
    # -----------------------------------------------------------

    {
        "rule_id": "DQ-V001",
        "dimension": "Validity",
        "table_name": "finished_product_type",
        "column_name": "mts_mto",
        "rule_type": "DOMAIN",
        "severity": "WARNING",
        "description":
            "MTS/MTO classification must remain within the observed "
            "source domain: MTS or MTO."
    },

    {
        "rule_id": "DQ-V002",
        "dimension": "Validity",
        "table_name": "product_production_alternative",
        "column_name": "main_alternative",
        "rule_type": "DOMAIN",
        "severity": "WARNING",
        "description":
            "Main-alternative source code must remain within the "
            "validated source domain: 1 or 2."
    },

    {
        "rule_id": "DQ-V003",
        "dimension": "Validity",
        "table_name": "customer_order",
        "column_name": "planning_horizon_status",
        "rule_type": "DOMAIN",
        "severity": "WARNING",
        "description":
            "Planning-horizon status must contain only the controlled "
            "Silver classifications."
    },

    {
        "rule_id": "DQ-V004",
        "dimension": "Validity",
        "table_name": "planning_periods",
        "column_name": "period",
        "rule_type": "SEQUENCE",
        "severity": "CRITICAL",
        "description":
            "Planning periods must remain a complete sequence from 1 "
            "through the configured number of periods."
    }
]


# ---------------------------------------------------------------
# 2. Validate registry metadata
# ---------------------------------------------------------------

rule_ids = [rule["rule_id"] for rule in dq_rules]

duplicate_rule_ids = (
    len(rule_ids) - len(set(rule_ids))
)

dimensions = sorted(
    set(rule["dimension"] for rule in dq_rules)
)

critical_rules = sum(
    rule["severity"] == "CRITICAL"
    for rule in dq_rules
)

warning_rules = sum(
    rule["severity"] == "WARNING"
    for rule in dq_rules
)


# ---------------------------------------------------------------
# 3. Display registry
# ---------------------------------------------------------------

for rule in dq_rules:

    print(
        f"{rule['rule_id']:<8} | "
        f"{rule['dimension']:<22} | "
        f"{rule['table_name']:<32} | "
        f"{rule['column_name']:<30} | "
        f"{rule['rule_type']:<12} | "
        f"{rule['severity']}"
    )


# ---------------------------------------------------------------
# 4. Registry summary
# ---------------------------------------------------------------

print("=" * 120)

print(f"Registered DQ rules:       {len(dq_rules)}")
print(f"Quality dimensions:        {len(dimensions)}")
print(f"Dimensions:                {', '.join(dimensions)}")
print(f"CRITICAL rules:            {critical_rules}")
print(f"WARNING rules:             {warning_rules}")
print(f"Duplicate rule IDs:        {duplicate_rule_ids}")

if duplicate_rule_ids == 0:
    print(
        "RULE REGISTRY STATUS: VALID — "
        "formal data-quality controls are defined "
        "and ready for execution."
    )
else:
    raise ValueError(
        "RULE REGISTRY INVALID: duplicate rule IDs detected."
    )

DATA QUALITY VALIDATION — RULE REGISTRY
DQ-C001  | Completeness           | customer_order                   | order_id                       | NOT_NULL     | CRITICAL
DQ-C002  | Completeness           | customer_order                   | finished_product               | NOT_NULL     | CRITICAL
DQ-C003  | Completeness           | demand_plan                      | finished_product               | NOT_NULL     | CRITICAL
DQ-C004  | Completeness           | demand_plan                      | period                         | NOT_NULL     | CRITICAL
DQ-U001  | Uniqueness             | customer_order                   | order_id                       | UNIQUE       | CRITICAL
DQ-U002  | Uniqueness             | finished_product_type            | finished_product               | UNIQUE       | CRITICAL
DQ-U003  | Uniqueness             | machine                          | machine                        | UNIQUE       | CRITICAL
DQ-U004  | Uniqueness             | packing_machine             

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Data Quality Validation — Step 03: Configure Rule Execution Parameters
#
# Purpose:
# Define the technical parameters required to execute the
# formal data-quality rules registered in Step 02.
#
# No Silver source data is modified.

print("DATA QUALITY VALIDATION — RULE EXECUTION PARAMETERS")
print("=" * 120)


# ---------------------------------------------------------------
# 1. Foreign-key relationship parameters
# ---------------------------------------------------------------

foreign_key_config = {

    "DQ-R001": {
        "parent_table": "finished_product_type",
        "parent_column": "finished_product"
    },

    "DQ-R002": {
        "parent_table": "finished_product_type",
        "parent_column": "finished_product"
    },

    "DQ-R003": {
        "parent_table": "machine",
        "parent_column": "machine"
    },

    "DQ-R004": {
        "parent_table": "packing_machine",
        "parent_column": "packing_machine"
    },

    "DQ-R005": {
        "parent_table": "planning_periods",
        "parent_column": "period"
    }
}


# ---------------------------------------------------------------
# 2. Controlled-domain parameters
# ---------------------------------------------------------------

domain_config = {

    "DQ-V001": {
        "allowed_values": ["MTO", "MTS"]
    },

    "DQ-V002": {
        "allowed_values": [1, 2]
    },

    "DQ-V003": {
        "allowed_values": [
            "Before horizon start",
            "Within horizon",
            "Beyond horizon end"
        ]
    }
}


# ---------------------------------------------------------------
# 3. Sequence-rule parameters
# ---------------------------------------------------------------

sequence_config = {

    "DQ-V004": {
        "parameter_table": "planning_parameters",
        "parameter_name": "Number of periods",
        "sequence_start": 1
    }
}


# ---------------------------------------------------------------
# 4. Validate registry-to-configuration coverage
# ---------------------------------------------------------------

# ---------------------------------------------------------------
# 4. Expected parameterized rule IDs
# ---------------------------------------------------------------
#
# These IDs correspond to the parameterized controls formally
# defined in the Step 02 rule registry.
#
# Keeping this validation self-contained prevents notebook-session
# resets from causing a NameError simply because dq_rules is no
# longer held in Python memory.

foreign_key_rule_ids = {
    "DQ-R001",
    "DQ-R002",
    "DQ-R003",
    "DQ-R004",
    "DQ-R005"
}

domain_rule_ids = {
    "DQ-V001",
    "DQ-V002",
    "DQ-V003"
}

sequence_rule_ids = {
    "DQ-V004"
}


fk_config_status = (
    "PASS"
    if foreign_key_rule_ids == set(foreign_key_config.keys())
    else "FAIL"
)

domain_config_status = (
    "PASS"
    if domain_rule_ids == set(domain_config.keys())
    else "FAIL"
)

sequence_config_status = (
    "PASS"
    if sequence_rule_ids == set(sequence_config.keys())
    else "FAIL"
)


# ---------------------------------------------------------------
# 5. Display configuration
# ---------------------------------------------------------------

print("FOREIGN-KEY CONFIGURATION")
print("-" * 120)

for rule_id, config in foreign_key_config.items():

    print(
        f"{rule_id:<8} | "
        f"Parent: "
        f"{config['parent_table']}.{config['parent_column']}"
    )


print()
print("DOMAIN CONFIGURATION")
print("-" * 120)

for rule_id, config in domain_config.items():

    print(
        f"{rule_id:<8} | "
        f"Allowed values: {config['allowed_values']}"
    )


print()
print("SEQUENCE CONFIGURATION")
print("-" * 120)

for rule_id, config in sequence_config.items():

    print(
        f"{rule_id:<8} | "
        f"Parameter source: "
        f"{config['parameter_table']} | "
        f"Parameter: {config['parameter_name']} | "
        f"Start: {config['sequence_start']}"
    )


# ---------------------------------------------------------------
# 6. Configuration summary
# ---------------------------------------------------------------

print("=" * 120)

print(
    f"Foreign-key rules configured: "
    f"{len(foreign_key_config)} / "
    f"{len(foreign_key_rule_ids)} | "
    f"{fk_config_status}"
)

print(
    f"Domain rules configured:      "
    f"{len(domain_config)} / "
    f"{len(domain_rule_ids)} | "
    f"{domain_config_status}"
)

print(
    f"Sequence rules configured:    "
    f"{len(sequence_config)} / "
    f"{len(sequence_rule_ids)} | "
    f"{sequence_config_status}"
)


if all(
    status == "PASS"
    for status in [
        fk_config_status,
        domain_config_status,
        sequence_config_status
    ]
):
    print(
        "RULE CONFIGURATION STATUS: VALID — "
        "all parameterized DQ rules are ready for execution."
    )

else:
    raise ValueError(
        "RULE CONFIGURATION INVALID: "
        "one or more registered rules are missing "
        "execution parameters."
    )

DATA QUALITY VALIDATION — RULE EXECUTION PARAMETERS
FOREIGN-KEY CONFIGURATION
------------------------------------------------------------------------------------------------------------------------
DQ-R001  | Parent: finished_product_type.finished_product
DQ-R002  | Parent: finished_product_type.finished_product
DQ-R003  | Parent: machine.machine
DQ-R004  | Parent: packing_machine.packing_machine
DQ-R005  | Parent: planning_periods.period

DOMAIN CONFIGURATION
------------------------------------------------------------------------------------------------------------------------
DQ-V001  | Allowed values: ['MTO', 'MTS']
DQ-V002  | Allowed values: [1, 2]
DQ-V003  | Allowed values: ['Before horizon start', 'Within horizon', 'Beyond horizon end']

SEQUENCE CONFIGURATION
------------------------------------------------------------------------------------------------------------------------
DQ-V004  | Parameter source: planning_parameters | Parameter: Number of periods | Start: 1
Foreign-k

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Data Quality Validation — Step 04: Execute Data Quality Rule Engine
#
# Purpose:
# Execute the 18 formal DQ controls against persisted Silver
# Delta tables and generate standardized PASS / FAIL results.
#
# This step is self-contained so it remains executable after
# a Databricks Python-session reset.
#
# No Silver source data is modified.

from pyspark.sql import functions as F

print("DATA QUALITY VALIDATION — RULE EXECUTION ENGINE")
print("=" * 140)


# ---------------------------------------------------------------
# 1. Source configuration
# ---------------------------------------------------------------

silver_catalog = "workspace"
silver_schema = "silver_sop"

def silver_table(table_name):
    return spark.table(
        f"{silver_catalog}.{silver_schema}.{table_name}"
    )


# ---------------------------------------------------------------
# 2. Reconstruct formal DQ rule registry
# ---------------------------------------------------------------

dq_rules = [

    # COMPLETENESS
    ("DQ-C001", "Completeness", "customer_order",
     "order_id", "NOT_NULL", "CRITICAL"),

    ("DQ-C002", "Completeness", "customer_order",
     "finished_product", "NOT_NULL", "CRITICAL"),

    ("DQ-C003", "Completeness", "demand_plan",
     "finished_product", "NOT_NULL", "CRITICAL"),

    ("DQ-C004", "Completeness", "demand_plan",
     "period", "NOT_NULL", "CRITICAL"),


    # UNIQUENESS
    ("DQ-U001", "Uniqueness", "customer_order",
     "order_id", "UNIQUE", "CRITICAL"),

    ("DQ-U002", "Uniqueness", "finished_product_type",
     "finished_product", "UNIQUE", "CRITICAL"),

    ("DQ-U003", "Uniqueness", "machine",
     "machine", "UNIQUE", "CRITICAL"),

    ("DQ-U004", "Uniqueness", "packing_machine",
     "packing_machine", "UNIQUE", "CRITICAL"),

    ("DQ-U005", "Uniqueness", "planning_periods",
     "period", "UNIQUE", "CRITICAL"),


    # REFERENTIAL INTEGRITY
    ("DQ-R001", "Referential Integrity", "customer_order",
     "finished_product", "FOREIGN_KEY", "CRITICAL"),

    ("DQ-R002", "Referential Integrity", "demand_plan",
     "finished_product", "FOREIGN_KEY", "CRITICAL"),

    ("DQ-R003", "Referential Integrity", "machine_capacity",
     "machine", "FOREIGN_KEY", "CRITICAL"),

    ("DQ-R004", "Referential Integrity", "packing_capacity",
     "packing_machine", "FOREIGN_KEY", "CRITICAL"),

    ("DQ-R005", "Referential Integrity", "demand_plan",
     "period", "FOREIGN_KEY", "CRITICAL"),


    # VALIDITY
    ("DQ-V001", "Validity", "finished_product_type",
     "mts_mto", "DOMAIN", "WARNING"),

    ("DQ-V002", "Validity", "product_production_alternative",
     "main_alternative", "DOMAIN", "WARNING"),

    ("DQ-V003", "Validity", "customer_order",
     "planning_horizon_status", "DOMAIN", "WARNING"),

    ("DQ-V004", "Validity", "planning_periods",
     "period", "SEQUENCE", "CRITICAL")
]


# ---------------------------------------------------------------
# 3. Execution parameters
# ---------------------------------------------------------------

foreign_key_config = {

    "DQ-R001":
        ("finished_product_type", "finished_product"),

    "DQ-R002":
        ("finished_product_type", "finished_product"),

    "DQ-R003":
        ("machine", "machine"),

    "DQ-R004":
        ("packing_machine", "packing_machine"),

    "DQ-R005":
        ("planning_periods", "period")
}


domain_config = {

    "DQ-V001": ["MTO", "MTS"],

    "DQ-V002": [1, 2],

    "DQ-V003": [
        "Before horizon start",
        "Within horizon",
        "Beyond horizon end"
    ]
}


# ---------------------------------------------------------------
# 4. Retrieve configured number of planning periods
# ---------------------------------------------------------------

number_of_periods_row = (
    silver_table("planning_parameters")
    .filter(
        F.col("parameter_name") == "Number of periods"
    )
    .select("parameter_value")
    .first()
)

if number_of_periods_row is None:
    raise ValueError(
        "Unable to retrieve 'Number of periods' "
        "from planning_parameters."
    )

number_of_periods = int(
    float(number_of_periods_row["parameter_value"])
)


# ---------------------------------------------------------------
# 5. Execute rules
# ---------------------------------------------------------------

dq_results = []


for (
    rule_id,
    dimension,
    table_name,
    column_name,
    rule_type,
    severity
) in dq_rules:

    df = silver_table(table_name)

    records_tested = df.count()
    violation_count = 0


    # -----------------------------------------------------------
    # NOT NULL
    # -----------------------------------------------------------

    if rule_type == "NOT_NULL":

        violation_count = (
            df
            .filter(F.col(column_name).isNull())
            .count()
        )


    # -----------------------------------------------------------
    # UNIQUE
    #
    # Count records participating in duplicated key values.
    # Example:
    # key A appears 3 times -> 3 violating records.
    # -----------------------------------------------------------

    elif rule_type == "UNIQUE":

        duplicate_keys = (
            df
            .groupBy(column_name)
            .count()
            .filter(F.col("count") > 1)
        )

        violation_count = (
            duplicate_keys
            .agg(
                F.coalesce(
                    F.sum("count"),
                    F.lit(0)
                ).alias("violations")
            )
            .first()["violations"]
        )

        violation_count = int(violation_count)


    # -----------------------------------------------------------
    # FOREIGN KEY
    #
    # Count child records whose non-null key has no match
    # in the configured parent table.
    # -----------------------------------------------------------

    elif rule_type == "FOREIGN_KEY":

        parent_table, parent_column = (
            foreign_key_config[rule_id]
        )

        parent_df = (
            silver_table(parent_table)
            .select(
                F.col(parent_column).alias("_parent_key")
            )
            .distinct()
        )

        violation_count = (
            df
            .filter(F.col(column_name).isNotNull())
            .join(
                parent_df,
                F.col(column_name)
                == F.col("_parent_key"),
                "left_anti"
            )
            .count()
        )


    # -----------------------------------------------------------
    # DOMAIN
    # -----------------------------------------------------------

    elif rule_type == "DOMAIN":

        allowed_values = domain_config[rule_id]

        violation_count = (
            df
            .filter(
                F.col(column_name).isNull()
                |
                (~F.col(column_name).isin(allowed_values))
            )
            .count()
        )


    # -----------------------------------------------------------
    # SEQUENCE
    #
    # Planning periods must exactly equal:
    # 1, 2, ..., configured Number of periods.
    # -----------------------------------------------------------

    elif rule_type == "SEQUENCE":

        actual_periods = {
            int(row[column_name])
            for row in (
                df
                .select(column_name)
                .where(F.col(column_name).isNotNull())
                .distinct()
                .collect()
            )
        }

        expected_periods = set(
            range(1, number_of_periods + 1)
        )

        missing_periods = (
            expected_periods - actual_periods
        )

        unexpected_periods = (
            actual_periods - expected_periods
        )

        violation_count = (
            len(missing_periods)
            + len(unexpected_periods)
        )


    else:
        raise ValueError(
            f"Unsupported rule type: {rule_type}"
        )


    status = (
        "PASS"
        if violation_count == 0
        else "FAIL"
    )


    dq_results.append({
        "rule_id": rule_id,
        "dimension": dimension,
        "table_name": table_name,
        "column_name": column_name,
        "rule_type": rule_type,
        "severity": severity,
        "records_tested": records_tested,
        "violation_count": violation_count,
        "status": status
    })


# ---------------------------------------------------------------
# 6. Display individual rule results
# ---------------------------------------------------------------

for result in dq_results:

    print(
        f"{result['rule_id']:<8} | "
        f"{result['dimension']:<22} | "
        f"{result['table_name']:<30} | "
        f"{result['rule_type']:<12} | "
        f"{result['severity']:<8} | "
        f"Tested: {result['records_tested']:<5} | "
        f"Violations: {result['violation_count']:<5} | "
        f"{result['status']}"
    )


# ---------------------------------------------------------------
# 7. Rule-engine summary
# ---------------------------------------------------------------

total_rules = len(dq_results)

passed_rules = sum(
    result["status"] == "PASS"
    for result in dq_results
)

failed_rules = sum(
    result["status"] == "FAIL"
    for result in dq_results
)

critical_failures = sum(
    result["status"] == "FAIL"
    and result["severity"] == "CRITICAL"
    for result in dq_results
)

warning_failures = sum(
    result["status"] == "FAIL"
    and result["severity"] == "WARNING"
    for result in dq_results
)

total_violations = sum(
    result["violation_count"]
    for result in dq_results
)


print("=" * 140)

print(f"DQ rules executed:        {total_rules}")
print(f"Rules passed:             {passed_rules}")
print(f"Rules failed:             {failed_rules}")
print(f"Critical failures:        {critical_failures}")
print(f"Warning failures:         {warning_failures}")
print(f"Total detected violations:{total_violations}")

print("-" * 140)


if critical_failures == 0:

    print(
        "DATA QUALITY GATE: PASS — "
        "no critical data-quality violations detected."
    )

else:

    print(
        "DATA QUALITY GATE: FAIL — "
        f"{critical_failures} critical rule(s) failed."
    )

DATA QUALITY VALIDATION — RULE EXECUTION ENGINE
DQ-C001  | Completeness           | customer_order                 | NOT_NULL     | CRITICAL | Tested: 61    | Violations: 0     | PASS
DQ-C002  | Completeness           | customer_order                 | NOT_NULL     | CRITICAL | Tested: 61    | Violations: 0     | PASS
DQ-C003  | Completeness           | demand_plan                    | NOT_NULL     | CRITICAL | Tested: 246   | Violations: 0     | PASS
DQ-C004  | Completeness           | demand_plan                    | NOT_NULL     | CRITICAL | Tested: 246   | Violations: 0     | PASS
DQ-U001  | Uniqueness             | customer_order                 | UNIQUE       | CRITICAL | Tested: 61    | Violations: 0     | PASS
DQ-U002  | Uniqueness             | finished_product_type          | UNIQUE       | CRITICAL | Tested: 67    | Violations: 0     | PASS
DQ-U003  | Uniqueness             | machine                        | UNIQUE       | CRITICAL | Tested: 39    | Violations: 0     | PASS


In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Data Quality Validation — Step 05: Build DQ Results DataFrame
#
# Purpose:
# Convert rule-engine output into a structured Spark DataFrame
# suitable for audit reporting, SQL analysis, monitoring,
# and later Delta persistence.
#
# No Silver source data is modified.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType
)

print("DATA QUALITY VALIDATION — RESULTS DATAFRAME")
print("=" * 120)


# ---------------------------------------------------------------
# 1. Verify Step 04 execution results are available
# ---------------------------------------------------------------

if "dq_results" not in globals():
    raise RuntimeError(
        "dq_results is not available in the current Python session. "
        "Run Step 04 before executing Step 05."
    )

if len(dq_results) != 18:
    raise ValueError(
        f"Expected 18 DQ execution results, "
        f"but found {len(dq_results)}."
    )


# ---------------------------------------------------------------
# 2. Define explicit audit-result schema
# ---------------------------------------------------------------

dq_result_schema = StructType([

    StructField("rule_id", StringType(), False),

    StructField("dimension", StringType(), False),

    StructField("table_name", StringType(), False),

    StructField("column_name", StringType(), False),

    StructField("rule_type", StringType(), False),

    StructField("severity", StringType(), False),

    StructField("records_tested", LongType(), False),

    StructField("violation_count", LongType(), False),

    StructField("status", StringType(), False)
])


# ---------------------------------------------------------------
# 3. Convert Python rule results to Spark DataFrame
# ---------------------------------------------------------------

dq_results_df = spark.createDataFrame(
    dq_results,
    schema=dq_result_schema
)


# ---------------------------------------------------------------
# 4. Add audit metadata
# ---------------------------------------------------------------

dq_results_df = (
    dq_results_df

    .withColumn(
        "source_layer",
        F.lit("Silver")
    )

    .withColumn(
        "source_schema",
        F.lit("workspace.silver_sop")
    )

    .withColumn(
        "validation_timestamp",
        F.current_timestamp()
    )
)


# ---------------------------------------------------------------
# 5. Standardize output column order
# ---------------------------------------------------------------

dq_results_df = dq_results_df.select(

    "rule_id",
    "dimension",
    "table_name",
    "column_name",
    "rule_type",
    "severity",
    "records_tested",
    "violation_count",
    "status",
    "source_layer",
    "source_schema",
    "validation_timestamp"
)


# ---------------------------------------------------------------
# 6. Validate DataFrame structure
# ---------------------------------------------------------------

result_row_count = dq_results_df.count()

distinct_rule_count = (
    dq_results_df
    .select("rule_id")
    .distinct()
    .count()
)

failed_rule_count = (
    dq_results_df
    .filter(F.col("status") == "FAIL")
    .count()
)

critical_failure_count = (
    dq_results_df
    .filter(
        (F.col("status") == "FAIL")
        &
        (F.col("severity") == "CRITICAL")
    )
    .count()
)


row_count_status = (
    "PASS"
    if result_row_count == 18
    else "FAIL"
)

rule_uniqueness_status = (
    "PASS"
    if distinct_rule_count == 18
    else "FAIL"
)


# ---------------------------------------------------------------
# 7. Display structured results
# ---------------------------------------------------------------

display(
    dq_results_df
    .orderBy("rule_id")
)


# ---------------------------------------------------------------
# 8. Audit DataFrame summary
# ---------------------------------------------------------------

print("=" * 120)

print(
    f"DQ result rows:             "
    f"{result_row_count} | "
    f"{row_count_status}"
)

print(
    f"Distinct rule IDs:          "
    f"{distinct_rule_count} | "
    f"{rule_uniqueness_status}"
)

print(
    f"Failed rules:               "
    f"{failed_rule_count}"
)

print(
    f"Critical failures:          "
    f"{critical_failure_count}"
)

print(
    f"Audit metadata columns:     "
    f"source_layer, source_schema, "
    f"validation_timestamp"
)

print("-" * 120)


if (
    row_count_status == "PASS"
    and rule_uniqueness_status == "PASS"
):
    print(
        "DQ RESULTS DATAFRAME STATUS: VALID — "
        "18 auditable rule-execution records are "
        "ready for persistence."
    )

else:
    raise ValueError(
        "DQ RESULTS DATAFRAME VALIDATION FAILED."
    )

DATA QUALITY VALIDATION — RESULTS DATAFRAME


rule_id,dimension,table_name,column_name,rule_type,severity,records_tested,violation_count,status,source_layer,source_schema,validation_timestamp
DQ-C001,Completeness,customer_order,order_id,NOT_NULL,CRITICAL,61,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:27:32.988Z
DQ-C002,Completeness,customer_order,finished_product,NOT_NULL,CRITICAL,61,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:27:32.988Z
DQ-C003,Completeness,demand_plan,finished_product,NOT_NULL,CRITICAL,246,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:27:32.988Z
DQ-C004,Completeness,demand_plan,period,NOT_NULL,CRITICAL,246,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:27:32.988Z
DQ-R001,Referential Integrity,customer_order,finished_product,FOREIGN_KEY,CRITICAL,61,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:27:32.988Z
DQ-R002,Referential Integrity,demand_plan,finished_product,FOREIGN_KEY,CRITICAL,246,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:27:32.988Z
DQ-R003,Referential Integrity,machine_capacity,machine,FOREIGN_KEY,CRITICAL,312,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:27:32.988Z
DQ-R004,Referential Integrity,packing_capacity,packing_machine,FOREIGN_KEY,CRITICAL,56,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:27:32.988Z
DQ-R005,Referential Integrity,demand_plan,period,FOREIGN_KEY,CRITICAL,246,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:27:32.988Z
DQ-U001,Uniqueness,customer_order,order_id,UNIQUE,CRITICAL,61,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:27:32.988Z


DQ result rows:             18 | PASS
Distinct rule IDs:          18 | PASS
Failed rules:               0
Critical failures:          0
Audit metadata columns:     source_layer, source_schema, validation_timestamp
------------------------------------------------------------------------------------------------------------------------
DQ RESULTS DATAFRAME STATUS: VALID — 18 auditable rule-execution records are ready for persistence.


In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Data Quality Validation — Step 06: Persist DQ Audit Results
#
# Purpose:
# Persist the structured DQ execution results as a managed
# Delta audit table for traceability and future monitoring.
#
# Silver source data is not modified.

from pyspark.sql import functions as F

print("DATA QUALITY VALIDATION — AUDIT RESULT PERSISTENCE")
print("=" * 120)


# ---------------------------------------------------------------
# 1. Target configuration
# ---------------------------------------------------------------

dq_catalog = "workspace"
dq_schema = "data_quality_sop"
dq_table = "dq_rule_results"

dq_full_table_name = (
    f"{dq_catalog}.{dq_schema}.{dq_table}"
)


# ---------------------------------------------------------------
# 2. Verify execution-result DataFrame is available
# ---------------------------------------------------------------

if "dq_results_df" not in globals():
    raise RuntimeError(
        "dq_results_df is not available in the current "
        "Python session. Run Steps 04 and 05 before Step 06."
    )


current_run_rows = dq_results_df.count()

if current_run_rows != 18:
    raise ValueError(
        f"Expected 18 DQ results for the current run, "
        f"but found {current_run_rows}."
    )


# ---------------------------------------------------------------
# 3. Create dedicated DQ audit schema
# ---------------------------------------------------------------

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{dq_catalog}.{dq_schema}"
)

print(
    f"DQ audit schema ready: "
    f"{dq_catalog}.{dq_schema}"
)


# ---------------------------------------------------------------
# 4. Add a validation-run identifier
#
# All 18 records from this execution receive the same run ID.
# ---------------------------------------------------------------

validation_run_id = (
    spark.range(1)
    .select(
        F.date_format(
            F.current_timestamp(),
            "yyyyMMdd_HHmmss_SSS"
        ).alias("validation_run_id")
    )
    .first()["validation_run_id"]
)

dq_results_persist_df = (
    dq_results_df
    .withColumn(
        "validation_run_id",
        F.lit(validation_run_id)
    )
    .select(
        "validation_run_id",
        "rule_id",
        "dimension",
        "table_name",
        "column_name",
        "rule_type",
        "severity",
        "records_tested",
        "violation_count",
        "status",
        "source_layer",
        "source_schema",
        "validation_timestamp"
    )
)


# ---------------------------------------------------------------
# 5. Capture table state before persistence
# ---------------------------------------------------------------

table_existed_before = spark.catalog.tableExists(
    dq_full_table_name
)

if table_existed_before:

    rows_before = (
        spark.table(dq_full_table_name)
        .count()
    )

else:

    rows_before = 0


# ---------------------------------------------------------------
# 6. Persist current validation run
#
# Append mode preserves historical validation runs.
# ---------------------------------------------------------------

(
    dq_results_persist_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(dq_full_table_name)
)


# ---------------------------------------------------------------
# 7. Read persisted audit table back from Delta
# ---------------------------------------------------------------

persisted_dq_df = spark.table(
    dq_full_table_name
)

rows_after = persisted_dq_df.count()

current_run_persisted_rows = (
    persisted_dq_df
    .filter(
        F.col("validation_run_id")
        == validation_run_id
    )
    .count()
)


# ---------------------------------------------------------------
# 8. Persistence reconciliation
# ---------------------------------------------------------------

expected_rows_after = (
    rows_before + current_run_rows
)

persistence_status = (
    "PASS"
    if (
        rows_after == expected_rows_after
        and current_run_persisted_rows
        == current_run_rows
    )
    else "FAIL"
)


# ---------------------------------------------------------------
# 9. Display current persisted run
# ---------------------------------------------------------------

display(
    persisted_dq_df
    .filter(
        F.col("validation_run_id")
        == validation_run_id
    )
    .orderBy("rule_id")
)


# ---------------------------------------------------------------
# 10. Persistence summary
# ---------------------------------------------------------------

print("=" * 120)

print(
    f"Target Delta table:          "
    f"{dq_full_table_name}"
)

print(
    f"Validation run ID:           "
    f"{validation_run_id}"
)

print(
    f"Table existed before run:    "
    f"{table_existed_before}"
)

print(
    f"Rows before persistence:     "
    f"{rows_before}"
)

print(
    f"Current-run result rows:     "
    f"{current_run_rows}"
)

print(
    f"Current-run persisted rows:  "
    f"{current_run_persisted_rows}"
)

print(
    f"Rows after persistence:      "
    f"{rows_after}"
)

print(
    f"Expected rows after:         "
    f"{expected_rows_after}"
)

print(
    f"Persistence reconciliation:  "
    f"{persistence_status}"
)

print("-" * 120)


if persistence_status == "PASS":

    print(
        "DQ AUDIT PERSISTENCE STATUS: PASS — "
        "the current validation run was successfully "
        "stored as Delta audit history."
    )

else:

    raise ValueError(
        "DQ AUDIT PERSISTENCE FAILED: "
        "persisted record counts do not reconcile."
    )

DATA QUALITY VALIDATION — AUDIT RESULT PERSISTENCE
DQ audit schema ready: workspace.data_quality_sop


validation_run_id,rule_id,dimension,table_name,column_name,rule_type,severity,records_tested,violation_count,status,source_layer,source_schema,validation_timestamp
20260928_052946_692,DQ-C001,Completeness,customer_order,order_id,NOT_NULL,CRITICAL,61,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:29:50.918Z
20260928_052946_692,DQ-C002,Completeness,customer_order,finished_product,NOT_NULL,CRITICAL,61,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:29:50.918Z
20260928_052946_692,DQ-C003,Completeness,demand_plan,finished_product,NOT_NULL,CRITICAL,246,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:29:50.918Z
20260928_052946_692,DQ-C004,Completeness,demand_plan,period,NOT_NULL,CRITICAL,246,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:29:50.918Z
20260928_052946_692,DQ-R001,Referential Integrity,customer_order,finished_product,FOREIGN_KEY,CRITICAL,61,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:29:50.918Z
20260928_052946_692,DQ-R002,Referential Integrity,demand_plan,finished_product,FOREIGN_KEY,CRITICAL,246,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:29:50.918Z
20260928_052946_692,DQ-R003,Referential Integrity,machine_capacity,machine,FOREIGN_KEY,CRITICAL,312,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:29:50.918Z
20260928_052946_692,DQ-R004,Referential Integrity,packing_capacity,packing_machine,FOREIGN_KEY,CRITICAL,56,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:29:50.918Z
20260928_052946_692,DQ-R005,Referential Integrity,demand_plan,period,FOREIGN_KEY,CRITICAL,246,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:29:50.918Z
20260928_052946_692,DQ-U001,Uniqueness,customer_order,order_id,UNIQUE,CRITICAL,61,0,PASS,Silver,workspace.silver_sop,2026-09-28T05:29:50.918Z


Target Delta table:          workspace.data_quality_sop.dq_rule_results
Validation run ID:           20260928_052946_692
Table existed before run:    False
Rows before persistence:     0
Current-run result rows:     18
Current-run persisted rows:  18
Rows after persistence:      18
Expected rows after:         18
Persistence reconciliation:  PASS
------------------------------------------------------------------------------------------------------------------------
DQ AUDIT PERSISTENCE STATUS: PASS — the current validation run was successfully stored as Delta audit history.


In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Data Quality Validation — Step 07: Build DQ Scorecard & Quality Gate
#
# Purpose:
# Read the latest persisted DQ validation run from Delta and
# produce an auditable quality scorecard and release gate.
#
# The scorecard uses a transparent rule pass rate:
# passed rules / executed rules.
#
# No source or audit data is modified.

from pyspark.sql import functions as F

print("DATA QUALITY VALIDATION — SCORECARD & QUALITY GATE")
print("=" * 120)


# ---------------------------------------------------------------
# 1. Persisted audit source
# ---------------------------------------------------------------

dq_audit_table = (
    "workspace.data_quality_sop.dq_rule_results"
)

dq_audit_df = spark.table(dq_audit_table)


# ---------------------------------------------------------------
# 2. Identify latest validation run
# ---------------------------------------------------------------

latest_run_row = (
    dq_audit_df
    .agg(
        F.max("validation_timestamp")
        .alias("latest_validation_timestamp")
    )
    .first()
)

latest_validation_timestamp = (
    latest_run_row["latest_validation_timestamp"]
)

if latest_validation_timestamp is None:
    raise ValueError(
        "No persisted DQ validation runs were found."
    )


latest_run_id = (
    dq_audit_df
    .filter(
        F.col("validation_timestamp")
        == latest_validation_timestamp
    )
    .select("validation_run_id")
    .first()["validation_run_id"]
)


latest_dq_df = (
    dq_audit_df
    .filter(
        F.col("validation_run_id")
        == latest_run_id
    )
)


# ---------------------------------------------------------------
# 3. Calculate overall quality metrics
# ---------------------------------------------------------------

total_rules = latest_dq_df.count()

passed_rules = (
    latest_dq_df
    .filter(F.col("status") == "PASS")
    .count()
)

failed_rules = (
    latest_dq_df
    .filter(F.col("status") == "FAIL")
    .count()
)

critical_failures = (
    latest_dq_df
    .filter(
        (F.col("status") == "FAIL")
        &
        (F.col("severity") == "CRITICAL")
    )
    .count()
)

warning_failures = (
    latest_dq_df
    .filter(
        (F.col("status") == "FAIL")
        &
        (F.col("severity") == "WARNING")
    )
    .count()
)

total_violations = (
    latest_dq_df
    .agg(
        F.coalesce(
            F.sum("violation_count"),
            F.lit(0)
        ).alias("total_violations")
    )
    .first()["total_violations"]
)

rule_pass_rate = (
    passed_rules / total_rules
    if total_rules > 0
    else 0
)


# ---------------------------------------------------------------
# 4. Determine formal quality gate
#
# Critical failures block downstream promotion.
# Warning failures require review but do not automatically
# block the dataset.
# ---------------------------------------------------------------

if critical_failures > 0:

    quality_gate = "FAIL"

elif warning_failures > 0:

    quality_gate = "PASS WITH WARNINGS"

else:

    quality_gate = "PASS"


# ---------------------------------------------------------------
# 5. Build dimension-level scorecard
# ---------------------------------------------------------------

dimension_scorecard_df = (
    latest_dq_df

    .groupBy("dimension")

    .agg(

        F.count("*")
        .alias("rules_executed"),

        F.sum(
            F.when(
                F.col("status") == "PASS",
                1
            ).otherwise(0)
        )
        .alias("rules_passed"),

        F.sum(
            F.when(
                F.col("status") == "FAIL",
                1
            ).otherwise(0)
        )
        .alias("rules_failed"),

        F.sum("violation_count")
        .alias("violations")
    )

    .withColumn(
        "rule_pass_rate_pct",

        F.round(
            (
                F.col("rules_passed")
                /
                F.col("rules_executed")
            ) * 100,
            2
        )
    )

    .orderBy("dimension")
)


# ---------------------------------------------------------------
# 6. Display dimension scorecard
# ---------------------------------------------------------------

display(dimension_scorecard_df)


# ---------------------------------------------------------------
# 7. Display failed rules, if any
# ---------------------------------------------------------------

failed_rules_df = (
    latest_dq_df
    .filter(F.col("status") == "FAIL")
    .select(
        "rule_id",
        "dimension",
        "table_name",
        "column_name",
        "rule_type",
        "severity",
        "records_tested",
        "violation_count"
    )
    .orderBy(
        F.desc("severity"),
        "rule_id"
    )
)

if failed_rules > 0:

    print()
    print("FAILED RULES REQUIRING REVIEW")
    print("-" * 120)

    display(failed_rules_df)

else:

    print()
    print(
        "FAILED RULES: None detected "
        "in the latest validation run."
    )


# ---------------------------------------------------------------
# 8. Scorecard summary
# ---------------------------------------------------------------

print()
print("=" * 120)

print(
    f"Audit source:              "
    f"{dq_audit_table}"
)

print(
    f"Latest validation run:     "
    f"{latest_run_id}"
)

print(
    f"Rules executed:            "
    f"{total_rules}"
)

print(
    f"Rules passed:              "
    f"{passed_rules}"
)

print(
    f"Rules failed:              "
    f"{failed_rules}"
)

print(
    f"Rule pass rate:            "
    f"{rule_pass_rate:.2%}"
)

print(
    f"Critical failures:         "
    f"{critical_failures}"
)

print(
    f"Warning failures:          "
    f"{warning_failures}"
)

print(
    f"Detected violations:       "
    f"{total_violations}"
)

print(
    f"Downstream quality gate:   "
    f"{quality_gate}"
)

print("=" * 120)


if quality_gate == "PASS":

    print(
        "QUALITY GATE STATUS: PASS — "
        "the latest Silver validation run contains "
        "no critical or warning rule failures."
    )

elif quality_gate == "PASS WITH WARNINGS":

    print(
        "QUALITY GATE STATUS: PASS WITH WARNINGS — "
        "no critical rules failed, but warning-level "
        "exceptions require review."
    )

else:

    print(
        "QUALITY GATE STATUS: FAIL — "
        "critical data-quality failures must be resolved "
        "before downstream promotion."
    )

DATA QUALITY VALIDATION — SCORECARD & QUALITY GATE


dimension,rules_executed,rules_passed,rules_failed,violations,rule_pass_rate_pct
Completeness,4,4,0,0,100.0
Referential Integrity,5,5,0,0,100.0
Uniqueness,5,5,0,0,100.0
Validity,4,4,0,0,100.0



FAILED RULES: None detected in the latest validation run.

Audit source:              workspace.data_quality_sop.dq_rule_results
Latest validation run:     20260928_052946_692
Rules executed:            18
Rules passed:              18
Rules failed:              0
Rule pass rate:            100.00%
Critical failures:         0
Warning failures:          0
Detected violations:       0
Downstream quality gate:   PASS
QUALITY GATE STATUS: PASS — the latest Silver validation run contains no critical or warning rule failures.


In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Data Quality Validation — Step 08: Completion Summary
#
# Purpose:
# Provide a concise final audit summary for the completed
# Data Quality Validation stage.
#
# This summary reads the persisted DQ audit history and does
# not modify Silver or audit data.

from pyspark.sql import functions as F

print("PROJECT #4 — DATA QUALITY VALIDATION COMPLETION SUMMARY")
print("=" * 100)

audit_table = "workspace.data_quality_sop.dq_rule_results"

audit_df = spark.table(audit_table)


# ---------------------------------------------------------------
# Identify latest persisted validation run
# ---------------------------------------------------------------

latest_timestamp = (
    audit_df
    .agg(
        F.max("validation_timestamp")
        .alias("latest_timestamp")
    )
    .first()["latest_timestamp"]
)

latest_run_id = (
    audit_df
    .filter(
        F.col("validation_timestamp") == latest_timestamp
    )
    .select("validation_run_id")
    .first()["validation_run_id"]
)

latest_run_df = (
    audit_df
    .filter(
        F.col("validation_run_id") == latest_run_id
    )
)


# ---------------------------------------------------------------
# Calculate final validation metrics
# ---------------------------------------------------------------

rules_executed = latest_run_df.count()

rules_passed = (
    latest_run_df
    .filter(F.col("status") == "PASS")
    .count()
)

rules_failed = (
    latest_run_df
    .filter(F.col("status") == "FAIL")
    .count()
)

critical_failures = (
    latest_run_df
    .filter(
        (F.col("status") == "FAIL")
        &
        (F.col("severity") == "CRITICAL")
    )
    .count()
)

warning_failures = (
    latest_run_df
    .filter(
        (F.col("status") == "FAIL")
        &
        (F.col("severity") == "WARNING")
    )
    .count()
)

total_violations = (
    latest_run_df
    .agg(
        F.coalesce(
            F.sum("violation_count"),
            F.lit(0)
        ).alias("violations")
    )
    .first()["violations"]
)

rule_pass_rate = (
    rules_passed / rules_executed
    if rules_executed > 0
    else 0
)

quality_gate = (
    "PASS"
    if critical_failures == 0
    and warning_failures == 0
    else
    "PASS WITH WARNINGS"
    if critical_failures == 0
    else
    "FAIL"
)


# ---------------------------------------------------------------
# Print completion summary
# ---------------------------------------------------------------

print(f"Silver source schema:        workspace.silver_sop")
print(f"DQ audit schema:             workspace.data_quality_sop")
print(f"DQ audit table:              dq_rule_results")
print(f"Latest validation run:       {latest_run_id}")

print()
print("FORMAL DATA QUALITY FRAMEWORK")
print("-" * 100)

print(f"Quality dimensions:          4")
print(f"Rules executed:              {rules_executed}")
print(f"Rules passed:                {rules_passed}")
print(f"Rules failed:                {rules_failed}")
print(f"Rule pass rate:              {rule_pass_rate:.2%}")

print()
print("QUALITY GATE")
print("-" * 100)

print(f"Critical failures:           {critical_failures}")
print(f"Warning failures:            {warning_failures}")
print(f"Detected violations:         {total_violations}")
print(f"Downstream quality gate:     {quality_gate}")

print()
print("AUDITABILITY")
print("-" * 100)

print("Persisted audit history:     YES")
print("Validation run identifier:   YES")
print("Validation timestamp:        YES")
print("Rule-level results:          YES")
print("Severity classification:     YES")
print("Violation counts:            YES")

print("=" * 100)

if quality_gate == "PASS":

    print(
        "DATA QUALITY VALIDATION STATUS: COMPLETE"
    )

    print(
        "The latest persisted Silver validation run "
        "passed all formal DQ controls and is eligible "
        "for downstream Gold-layer modeling."
    )

else:

    print(
        "DATA QUALITY VALIDATION STATUS: REVIEW REQUIRED"
    )

PROJECT #4 — DATA QUALITY VALIDATION COMPLETION SUMMARY
Silver source schema:        workspace.silver_sop
DQ audit schema:             workspace.data_quality_sop
DQ audit table:              dq_rule_results
Latest validation run:       20260928_052946_692

FORMAL DATA QUALITY FRAMEWORK
----------------------------------------------------------------------------------------------------
Quality dimensions:          4
Rules executed:              18
Rules passed:                18
Rules failed:                0
Rule pass rate:              100.00%

QUALITY GATE
----------------------------------------------------------------------------------------------------
Critical failures:           0
Warning failures:            0
Detected violations:         0
Downstream quality gate:     PASS

AUDITABILITY
----------------------------------------------------------------------------------------------------
Persisted audit history:     YES
Validation run identifier:   YES
Validation timestamp:     